In [179]:
import pandas as pd

# Define the data
data = {
    'customer_id': [1, 1, 2, 2, 3],
    'order_amount': [100, 200, 150, 50, 300],
    'order_date': ['2024-01-01', '2024-01-05', '2024-01-03', '2024-01-10', '2024-01-02']
}

# Create the DataFrame
df = pd.DataFrame(data)

# Convert 'order_date' to actual datetime objects
df['order_date'] = pd.to_datetime(df['order_date'])

print(df)

   customer_id  order_amount order_date
0            1           100 2024-01-01
1            1           200 2024-01-05
2            2           150 2024-01-03
3            2            50 2024-01-10
4            3           300 2024-01-02


Task 1 (Basic Filtering + Aggregation):

1. Find total order amount per customer

2. Return only customers with total > 200

In [180]:
df['total_order_amount'] = df.groupby('customer_id')['order_amount'].transform('sum')
df

,customer_id,order_amount,order_date,total_order_amount
0,1,100,2024-01-01,300
1,1,200,2024-01-05,300
2,2,150,2024-01-03,200
3,2,50,2024-01-10,200
4,3,300,2024-01-02,300


In [181]:
df.loc[df['total_order_amount'] > 200]

,customer_id,order_amount,order_date,total_order_amount
0,1,100,2024-01-01,300
1,1,200,2024-01-05,300
4,3,300,2024-01-02,300


Corrected code suggested in review.

- Transform returns row level output of the initial dataframe. It is not needed for the answer
- Think about how the stakeholders could easily read the output

In [182]:
result = (
    df.groupby('customer_id', as_index=False)['order_amount']
      .sum()
      .rename(columns={'order_amount': 'total_order_amount'})
)

result = result[result['total_order_amount'] > 200]
result

,customer_id,total_order_amount
0,1,300
2,3,300


Task 2 (Groupby + Ranking):

1. For each customer, find their largest order

2. Then return the top 2 customers based on that largest order

In [183]:
df.groupby('customer_id')['order_amount'].max()

customer_id
1    200
2    150
3    300
Name: order_amount, dtype: int64

In [184]:
df.groupby('customer_id')['order_amount'].max().sort_values(ascending = False).head(2)

customer_id
3    300
1    200
Name: order_amount, dtype: int64

In [185]:
df.groupby('customer_id', as_index=False)['order_amount'].max().rename(columns={'order_amount': 'largest_order'})

,customer_id,largest_order
0,1,200
1,2,150
2,3,300


Cleaner answer suggested in review.
- Returns a dataframe, 
- renamed columns, 
- production friendly

In [186]:
result = (
    df.groupby('customer_id', as_index=False)['order_amount']
      .max()
      .rename(columns={'order_amount': 'largest_order'})
      .sort_values(by='largest_order', ascending=False)
      .head(2)
)
result

,customer_id,largest_order
2,3,300
0,1,200


In [187]:
# Define the data in a dictionary
data = {
    'customer_id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie']
}

# Create the DataFrame
customers = pd.DataFrame(data)

# Display the table
print(customers)

# Define the orders data
orders_data = {
    'order_id': [101, 102, 103, 104],
    'customer_id': [1, 1, 2, 4],
    'amount': [100, 200, 50, 300]
}

# Create the DataFrame
orders = pd.DataFrame(orders_data)

print(orders)



   customer_id     name
0            1    Alice
1            2      Bob
2            3  Charlie
   order_id  customer_id  amount
0       101            1     100
1       102            1     200
2       103            2      50
3       104            4     300


Task 3 (Merge + Filtering):

- Join customers with orders (keep only valid matches)
- Calculate total order amount per customer
    
    Return:

        - customer_id

        - name

        - total_amount

- Only include customers with total_amount ≥ 150

In [188]:
merged = customers.merge(orders, on = 'customer_id', how = 'inner')
merged

,customer_id,name,order_id,amount
0,1,Alice,101,100
1,1,Alice,102,200
2,2,Bob,103,50


In [189]:
result = merged.groupby(['customer_id', 'name'], as_index=False)['amount'].sum()
result

,customer_id,name,amount
0,1,Alice,300
1,2,Bob,50


In [190]:
result = result.loc[result['amount'] >= 150 ]
result

,customer_id,name,amount
0,1,Alice,300


Task 4 (transform + ranking):

- For each customer, add a column:

       - running_total → cumulative sum of orders by date

- For each customer, rank their orders:

       - Highest order = rank 1

       - Use dense ranking (no gaps)

- Return only the top order per customer using your ranking



In [191]:
# Create the DataFrame
df = pd.DataFrame({
    'customer_id': [1, 1, 1, 2, 2],
    'order_amount': [100, 200, 50, 300, 100],
    'order_date': ['2024-01-01', '2024-01-05', '2024-01-07', '2024-01-02', '2024-01-06']
})

# Convert order_date to datetime objects for better analysis
df['order_date'] = pd.to_datetime(df['order_date'])

print(df)


   customer_id  order_amount order_date
0            1           100 2024-01-01
1            1           200 2024-01-05
2            1            50 2024-01-07
3            2           300 2024-01-02
4            2           100 2024-01-06


In [192]:
df = df.set_index('order_date')
df['cumulative_sum'] = df.groupby('customer_id')['order_amount'].cumsum()
df

,customer_id,order_amount,cumulative_sum
order_date,,,
2024-01-01,1,100,100
2024-01-05,1,200,300
2024-01-07,1,50,350
2024-01-02,2,300,300
2024-01-06,2,100,400


In [193]:
#df = df.drop('cumulative_sum', axis = 1)
df = df.reset_index('order_date')
df['rank_per_customer'] = (df.groupby('customer_id')['order_amount']
                           .rank(method='dense', ascending = False))
df

,order_date,customer_id,order_amount,cumulative_sum,rank_per_customer
0,2024-01-01,1,100,100,2.0
1,2024-01-05,1,200,300,1.0
2,2024-01-07,1,50,350,3.0
3,2024-01-02,2,300,300,1.0
4,2024-01-06,2,100,400,2.0


In [194]:
df = df.loc[df['rank_per_customer'] == 1]
df

,order_date,customer_id,order_amount,cumulative_sum,rank_per_customer
1,2024-01-05,1,200,300,1.0
3,2024-01-02,2,300,300,1.0


Cleaner answer suggested in review.

- setting date index is not necessary in this scenario
- sort data explicitly

In [195]:
# Step 1: Sort for correct running total
df = df.sort_values(['customer_id', 'order_date'])

# Step 2: Running total
df['running_total'] = df.groupby('customer_id')['order_amount'].cumsum()

# Step 3: Dense rank
df['rank'] = (
    df.groupby('customer_id')['order_amount']
      .rank(method='dense', ascending=False)
)

# Step 4: Top order per customer
top_orders = df[df['rank'] == 1]

In [196]:
# Define the data in a dictionary
data = {
    'customer_id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'David']
}

# Create the DataFrame
customers = pd.DataFrame(data)

# Display the table
print(customers)

# Define the orders data
orders_data = {
    'order_id': [101, 102, 103],
    'customer_id': [1, 1, 2],
    'amount': [100, 200, 50]
}

# Create the DataFrame
orders = pd.DataFrame(orders_data)

print(orders)

   customer_id     name
0            1    Alice
1            2      Bob
2            3  Charlie
3            4    David
   order_id  customer_id  amount
0       101            1     100
1       102            1     200
2       103            2      50


Task 5 (Merge + Missing Data):

- Return all customers (even those with no orders)

- Calculate total order amount per customer

        Customers with no orders should have total_amount = 0

        Output:

            - customer_id

            - name

            - total_amount

In [197]:
# Return all customers (even those with no orders)
merge_df = customers.merge(orders, on= 'customer_id', how= 'left').fillna(0)
total_df = merge_df.groupby(['customer_id', 'name'], as_index= False)['amount'].sum()
total_df.rename(columns = {'amount': 'total_amount'})

,customer_id,name,total_amount
0,1,Alice,300.0
1,2,Bob,50.0
2,3,Charlie,0.0
3,4,David,0.0


Solution suggested in review:

In [198]:
result = (
    customers.merge(orders, on='customer_id', how='left')
    .assign(amount=lambda x: x['amount'].fillna(0))
    .groupby(['customer_id', 'name'], as_index=False)['amount']
    .sum()
    .rename(columns={'amount': 'total_amount'})
)

In [199]:
# Create the DataFrame
data = {
    'customer_id': [1, 1, 2, 2, 2],
    'product': ['A', 'B', 'A', 'B', 'C'],
    'amount': [100, 200, 50, 150, 300]
}
df = pd.DataFrame(data)

Task 6 (Pivot / Reshape): 

- Transform into this format:

customer_id | A   | B   | C

1           | 100 | 200 | 0

2           | 50  | 150 | 300

        Missing values should be 0

- Add a column:

        total_spent = sum across products

- Return the top 1 customer based on total_spent

In [200]:
pivot_df = df.pivot_table(columns= 'product', index= 'customer_id', values= 'amount', fill_value= 0)
pivot_df['total_spent'] = pivot_df.sum(axis=1, numeric_only=True)
pivot_df

product,A,B,C,total_spent
customer_id,,,,
1,100,200,0,300
2,50,150,300,500


In [201]:
top_customer = pivot_df.sort_values('total_spent', ascending=False).head(1)
top_customer

product,A,B,C,total_spent
customer_id,,,,
2,50,150,300,500


TASK 7 (Second Highest per Group):

or each customer_id, return the second highest DISTINCT order_amount

In [229]:
# Define the data
data = {
    'customer_id': [1, 1, 1, 1, 2, 2, 2],
    'order_amount': [100, 200, 200, 50, 300, 100, 100]
}

# Create the DataFrame
df = pd.DataFrame(data)
df

,customer_id,order_amount
0,1,100
1,1,200
2,1,200
3,1,50
4,2,300
5,2,100
6,2,100


In [230]:
df = df.drop_duplicates()
df = df.sort_values(['customer_id', 'order_amount'], ascending = False)
df['rank'] = df.groupby('customer_id')['order_amount'].rank('dense', ascending=False)
df.loc[df['rank']== 2]

,customer_id,order_amount,rank
5,2,100,2.0
0,1,100,2.0


Points to fix:

- rank the specific column needed. 
- rename columns and make the output clear

In [ ]:
# solution provided in the review
df_unique = df.drop_duplicates()

df_unique['rank'] = (
    df_unique.groupby('customer_id')['order_amount']
             .rank(method='dense', ascending=False)
)

result = (
    df_unique[df_unique['rank'] == 2]
    [['customer_id', 'order_amount']]
    .rename(columns={'order_amount': 'second_highest'})
)